In [1]:
!pip install tensorboardX rdkit
!pip install rdkit-pypi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 2.0 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.1/33.1 MB 46.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.4/29.4 MB 52.1 MB/s eta 0:00:00:00:0100:01


In [2]:
import os
import torch
import torch.autograd as autograd
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data as Data
torch.manual_seed(8) # for reproduce


from sklearn.model_selection import LeaveOneOut
import time
from tqdm import tqdm
import numpy as np
import gc
import sys
sys.path.append('/content/drive/MyDrive/Colab Notebooks/Posdoc/Native_AFP/code/')
sys.setrecursionlimit(50000)
import pickle
torch.backends.cudnn.benchmark = True
torch.set_default_tensor_type('torch.cuda.FloatTensor')
# from tensorboardX import SummaryWriter
torch.nn.Module.dump_patches = True
import copy
import pandas as pd
#then import my own modules
from GCN import RelativeGCNModel,save_smiles_dicts, get_smiles_dicts, get_smiles_array, moltosvg_highlight
from rdkit import Chem
# from rdkit.Chem import AllChem
from rdkit.Chem import QED
from rdkit.Chem import rdMolDescriptors, MolSurf
from rdkit.Chem.Draw import SimilarityMaps
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import rdDepictor
from rdkit.Chem.Draw import rdMolDraw2D
%matplotlib inline
from numpy.polynomial.polynomial import polyfit
import matplotlib.pyplot as plt
from matplotlib import gridspec
import matplotlib.cm as cm
import matplotlib
import seaborn as sns; sns.set_style("darkgrid")
from IPython.display import SVG, display
from torch.utils.data import Dataset, TensorDataset, DataLoader
import uuid



import itertools
from sklearn.metrics import r2_score,mean_squared_error
import scipy
from itertools import combinations
from sklearn.model_selection import train_test_split

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler
import pickle
import os
import time
from rdkit import Chem
from sklearn.model_selection import LeaveOneOut
from collections import defaultdict


device= torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

/usr/local/lib/python3.11/dist-packages/torch/__init__.py:614: UserWarning: torch.set_default_tensor_type() is deprecated as of PyTorch 2.1, please use torch.set_default_dtype() and torch.set_default_device() as alternatives. (Triggered internally at ../torch/csrc/tensor/python_tensor.cpp:451.)
  _C._set_default_tensor_type(t)


cuda


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class RelativeGCNModel(nn.Module):
    def __init__(self, radius, T, input_feature_dim, input_bond_dim,
                 fingerprint_dim, output_units_num, p_dropout):
        super(RelativeGCNModel, self).__init__()
        
        self.radius = radius
        self.T = T
        
        # Increase the dimensionality of hidden layers
        hidden_dim = fingerprint_dim * 4  # Significantly larger hidden dimension
        
        self.atom_fc = nn.Sequential(
            nn.Linear(input_feature_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, fingerprint_dim)
        )
        
        self.neighbor_fc = nn.Sequential(
            nn.Linear(input_feature_dim + input_bond_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, fingerprint_dim)
        )
        
        # Increase the number and size of GCN layers
        self.gcn_layers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(fingerprint_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, fingerprint_dim)
            ) for _ in range(radius * 2)  # Double the number of GCN layers
        ])
        
        self.dropout = nn.Dropout(p=p_dropout)
        
        # More complex output layer for pairwise prediction
        self.output = nn.Sequential(
            nn.Linear(fingerprint_dim * 2, hidden_dim),  # Input is concatenated features of two molecules
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_units_num)
        )

    def forward(self, atom_list1, bond_list1, atom_degree_list1, bond_degree_list1, atom_mask1,
                atom_list2, bond_list2, atom_degree_list2, bond_degree_list2, atom_mask2):
        
        batch_size, mol_length, num_atom_feat = atom_list1.size()

        # Process first molecule
        atom_feature1, mol_feature1 = self.process_molecule(atom_list1, bond_list1, atom_degree_list1, bond_degree_list1, atom_mask1)
        
        # Process second molecule
        atom_feature2, mol_feature2 = self.process_molecule(atom_list2, bond_list2, atom_degree_list2, bond_degree_list2, atom_mask2)
        
        # Concatenate features of both molecules and predict pairwise difference
        pairwise_feature = torch.cat([mol_feature1, mol_feature2], dim=1)
        pairwise_prediction = self.output(self.dropout(pairwise_feature))
        
        return atom_feature1, atom_feature2, pairwise_prediction

    def process_molecule(self, atom_list, bond_list, atom_degree_list, bond_degree_list, atom_mask):
        batch_size, mol_length, num_atom_feat = atom_list.size()
        atom_mask = atom_mask.unsqueeze(2)
        
        #print("1. atom_list shape:", atom_list.shape)
        atom_feature = self.atom_fc(atom_list)
        #print("2. atom_feature shape after atom_fc:", atom_feature.shape)
        
        neighbor_feature = self.get_neighbor_feature(atom_list, bond_list, atom_degree_list, bond_degree_list, batch_size, mol_length)
        #print("3. neighbor_feature shape:", neighbor_feature.shape)
        
        for i, layer in enumerate(self.gcn_layers):
            neighbor_feature = layer(neighbor_feature)
           # print(f"4. neighbor_feature shape after gcn layer {i}:", neighbor_feature.shape)
            atom_feature = atom_feature + neighbor_feature
            atom_feature = self.dropout(atom_feature)
        #    print(f"5. atom_feature shape after layer {i}:", atom_feature.shape)
        
        mol_feature = torch.sum(F.relu(atom_feature) * atom_mask, dim=1)
       # print("6. mol_feature shape:", mol_feature.shape)
        return atom_feature, mol_feature

    

    def get_neighbor_feature(self, atom_list, bond_list, atom_degree_list, bond_degree_list, batch_size, mol_length):
        neighbor_feature = []
        for i in range(batch_size):
            atom_degrees = atom_degree_list[i]
            bond_degrees = bond_degree_list[i]
        
            # Ensure indices are within bounds
            atom_degrees = torch.clamp(atom_degrees, 0, mol_length - 1)
            bond_degrees = torch.clamp(bond_degrees, 0, bond_list.size(1) - 1)
        
        # No need to pad, use the original tensors
            atom_features = atom_list[i]
            bond_features = bond_list[i]
        
        # Gather neighbor features
            neighbor_atoms = atom_features[atom_degrees]
            neighbor_bonds = bond_features[bond_degrees]
            
        # Combine atom and bond features
            mol_neighbor_feature = torch.cat([neighbor_atoms, neighbor_bonds], dim=-1)
        
        # Apply neighbor_fc to each atom's neighborhood and sum
            mol_neighbor_feature = self.neighbor_fc(mol_neighbor_feature)
            mol_neighbor_feature = mol_neighbor_feature.sum(dim=1)
        
            neighbor_feature.append(mol_neighbor_feature)
    
        neighbor_feature = torch.stack(neighbor_feature, dim=0)
        return F.relu(neighbor_feature)
    


In [4]:
#?) pairs are directed or undirected?
#?) tergets correspond correctly? 
#?) 

def custom_cv_split(df):
    all_smiles = set()
    for pair in df['SMILES_pair']:
        smiles1, smiles2 = pair.split(',')
        all_smiles.add(smiles1)
        all_smiles.add(smiles2)
    
    n_splits = len(all_smiles)
    
    for test_smile in all_smiles:
        test_indices = []
        train_indices = []
        for idx, pair in enumerate(df['SMILES_pair']):
            smiles1, smiles2 = pair.split(',')
            if test_smile in (smiles1, smiles2):
                test_indices.append(idx)
            else:
                train_indices.append(idx)
        yield train_indices, test_indices, test_smile

#_____________________________________________________________________________________________________________________________________________ 
targets = ['H3K4', 'H3K4ac', 'H3K4me1', 'H3K4me2', 'H3K4me3', 'H3K9me3', 'H3R2me2a', 'H3R2me2s']
def validate(model, val_df, feature_dicts, loss_function, device, batch_size, return_predictions=False, test_smile=None):
    model.eval()
    total_loss = 0
    all_predictions = []
    all_true_values = []
    result_dict = {}

    with torch.no_grad():
        for i in range(0, len(val_df), batch_size):
            batch_df = val_df.iloc[i:i+batch_size]
            
            smile_pairs = batch_df.SMILES_pair.values
            y_val = batch_df[targets].values
            
            smiles_list1, smiles_list2 = zip(*[pair.split(',') for pair in smile_pairs])
            
            x_atom1, x_bonds1, x_atom_index1, x_bond_index1, x_mask1, _ = get_smiles_array(smiles_list1, feature_dicts)
            x_atom2, x_bonds2, x_atom_index2, x_bond_index2, x_mask2, _ = get_smiles_array(smiles_list2, feature_dicts)
            
            x_atom1, x_bonds1 = torch.Tensor(x_atom1).to(device), torch.Tensor(x_bonds1).to(device)
            x_atom_index1, x_bond_index1 = torch.LongTensor(x_atom_index1).to(device), torch.LongTensor(x_bond_index1).to(device)
            x_mask1 = torch.Tensor(x_mask1).to(device)
            
            x_atom2, x_bonds2 = torch.Tensor(x_atom2).to(device), torch.Tensor(x_bonds2).to(device)
            x_atom_index2, x_bond_index2 = torch.LongTensor(x_atom_index2).to(device), torch.LongTensor(x_bond_index2).to(device)
            x_mask2 = torch.Tensor(x_mask2).to(device)
            
            output_tuple = model(x_atom1, x_bonds1, x_atom_index1, x_bond_index1, x_mask1,
                                 x_atom2, x_bonds2, x_atom_index2, x_bond_index2, x_mask2)
            
            predictions = output_tuple[2]
            
            loss = loss_function(predictions, torch.Tensor(y_val).to(device))
            total_loss += loss.item()
            
            all_predictions.extend(predictions.cpu().numpy())
            all_true_values.extend(y_val)

            # Populate result_dict
            for j, smile_pair in enumerate(smile_pairs):
                smile1, smile2 = smile_pair.split(',')
                if smile1 == test_smile or smile2 == test_smile:
                    if test_smile not in result_dict:
                        result_dict[test_smile] = {}
                    result_dict[test_smile][smile_pair] = {}
                    for k, target in enumerate(targets):
                        result_dict[test_smile][smile_pair][target] = {
                            "actual": y_val[j][k],
                            "predicted": predictions[j][k].item()
                        }
    
    avg_loss = total_loss / (len(val_df) // batch_size + 1)
    r2 = r2_score(all_true_values, all_predictions)
    
    if return_predictions:
        if test_smile:
            return avg_loss, r2, all_true_values, all_predictions, result_dict
        else:
            print('NO TEST SMILE!!!!!')
            return avg_loss, r2, all_true_values, all_predictions, result_dict
    else:
        return avg_loss, r2



#_____________________________________________________________________________________________________________________________________________        

targets = ['H3K4', 'H3K4ac', 'H3K4me1', 'H3K4me2', 'H3K4me3', 'H3K9me3', 'H3R2me2a', 'H3R2me2s']
def prepare_model_and_data_for_relative_learning(raw_filename, target_name='Calx', targets=None, random_seed=888, batch_size=50, epochs=800, p_dropout=0.1, fingerprint_dim=210, weight_decay=5, learning_rate=3, radius=4, T=2, per_target_output_units_num=1):
    if targets is None:
        targets = ['H3K4', 'H3K4ac', 'H3K4me1', 'H3K4me2', 'H3K4me3', 'H3K9me3', 'H3R2me2a', 'H3R2me2s']

    feature_filename = raw_filename.replace('.csv', '.pickle')
    smiles_targets_df = pd.read_csv(raw_filename)

    smilesList = smiles_targets_df.SMILES.values
    #print("number of all smiles: ", len(smilesList))

    atom_num_dist = []
    remained_smiles = []
    canonical_smiles_list = []
    for smiles in smilesList:
        try:
            mol = Chem.MolFromSmiles(smiles)
            atom_num_dist.append(len(mol.GetAtoms()))
            remained_smiles.append(smiles)
            canonical_smiles_list.append(Chem.MolToSmiles(Chem.MolFromSmiles(smiles), isomericSmiles=True))
        except:
            print(smiles)
            pass
   # print("number of successfully processed smiles: ", len(remained_smiles))

    smiles_targets_df = smiles_targets_df[smiles_targets_df["SMILES"].isin(remained_smiles)]
    smiles_targets_df['cano_smiles'] = canonical_smiles_list

    if os.path.isfile(feature_filename):
        feature_dicts = pickle.load(open(feature_filename, "rb"))
    else:
        feature_dicts = save_smiles_dicts(smilesList, filename)
    
    feature_dicts = get_smiles_dicts(smilesList)
    #print (type(feature_dicts))
    remained_df = smiles_targets_df[smiles_targets_df["cano_smiles"].isin(feature_dicts['smiles_to_atom_mask'].keys())]

    # Create pairs of SMILES and calculate relative targets
    smile_pairs = []
    relative_targets = []
    
    for i in range(len(remained_df)):
        for j in range(len(remained_df)):
            if i != j:  # Exclude self-pairs
                smile1 = remained_df.iloc[i]['cano_smiles']
                smile2 = remained_df.iloc[j]['cano_smiles']
                target1 = remained_df.iloc[i][targets].values
                target2 = remained_df.iloc[j][targets].values
                
                # Calculate relative target (difference)
                rel_target = target1 - target2
                
                smile_pairs.append(f"{smile1},{smile2}")
                relative_targets.append(rel_target)

    # Create DataFrame
    df = pd.DataFrame(relative_targets, columns=targets)
    df['SMILES_pair'] = smile_pairs
    df = df[['SMILES_pair'] + targets]  # Reorder columns

    # Prepare input features for the model
    feature_data = []
    
    for smile_pair in smile_pairs:
        smile1, smile2 = smile_pair.split(',')
        x_atom1, x_bonds1, x_atom_index1, x_bond_index1, x_mask1, _ = get_smiles_array([smile1], feature_dicts)
        x_atom2, x_bonds2, x_atom_index2, x_bond_index2, x_mask2, _ = get_smiles_array([smile2], feature_dicts)
        
        feature_data.append({
            'x_atom1': x_atom1,
            'x_bonds1': x_bonds1,
            'x_atom_index1': x_atom_index1,
            'x_bond_index1': x_bond_index1,
            'x_mask1': x_mask1,
            'x_atom2': x_atom2,
            'x_bonds2': x_bonds2,
            'x_atom_index2': x_atom_index2,
            'x_bond_index2': x_bond_index2,
            'x_mask2': x_mask2
        })

    num_atom_features = feature_data[0]['x_atom1'].shape[-1]
    num_bond_features = feature_data[0]['x_bonds1'].shape[-1]

    loss_function = nn.MSELoss()
    model = RelativeGCNModel(radius, T, num_atom_features, num_bond_features, fingerprint_dim, len(targets), p_dropout)
    model.to(device)

    optimizer = optim.Adam(model.parameters(), 10**-learning_rate, weight_decay=10**-weight_decay)

    return model, optimizer, loss_function, df, feature_data, feature_filename,remained_df

# Example usage:
model, optimizer, loss_function, df, feature_data, feature_filename ,remained_df= prepare_model_and_data_for_relative_learning('/notebooks/data/cal_abs.csv')


In [5]:
if os.path.isfile(feature_filename):
    feature_dicts = pickle.load(open(feature_filename, "rb" ))
else:
    feature_dicts = save_smiles_dicts(smilesList,filename)
    
# feature_dicts = get_smiles_dicts(smilesList)

print(type(feature_dicts))
for index, row in df.iterrows():
    # Split the SMILES pair
    smile1, smile2 = row['SMILES_pair'].split(',')
    
    # Get features for smile1
    x_atom1, x_bonds1, x_atom_index1, x_bond_index1, x_mask1, _ = get_smiles_array([smile1], feature_dicts)

    # Get features for smile2
    x_atom2, x_bonds2, x_atom_index2, x_bond_index2, x_mask2, _ = get_smiles_array([smile2], feature_dicts)
print(x_atom2.shape)

<class 'dict'>
(1, 73, 39)


In [6]:


def train_and_evaluate(model, df, feature_dicts, optimizer, loss_function, num_epochs=100, batch_size=32, patience=10):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    fold_results = []
    all_true_values = []
    all_predictions = []
    final_dic= {}
    test_mses = [] 
    initial_state = {k: v.clone() for k, v in model.state_dict().items()}
    best_model_path = None
    
    
    n_splits = len(set(smiles for pair in df['SMILES_pair'] for smiles in pair.split(',')))
    
    for fold, (train_val_index, test_index, test_smile) in enumerate(custom_cv_split(df), 1):
        #print(f"\nFold {fold}/{n_splits}")
        #print(f"Testing SMILES: {test_smile}")
        
        # Reset the model for each fold
        model.load_state_dict(initial_state)
        
        # Reset optimizer state
        optimizer.state = defaultdict(dict)
        
        # Split the data into train+val and test
        train_val_df = df.iloc[train_val_index]
        test_df = df.iloc[test_index]
        
        # Check for test SMILES in train_val set
        train_val_smiles = set()
        for pair in train_val_df['SMILES_pair']:
            smiles1, smiles2 = pair.split(',')
            train_val_smiles.add(smiles1)
            train_val_smiles.add(smiles2)
        
        if test_smile in train_val_smiles:
            print(f"WARNING: Test SMILES {test_smile} found in train_val set!")
        #else:
            #print(f"Verification: Test SMILES {test_smile} not found in train_val set.")
        
        # Further split train_val into train and validation
        train_df, val_df = train_test_split(train_val_df, test_size=0.2, random_state=42)
        
        # Additional check for test SMILES in train and val sets
        train_smiles = set()
        for pair in train_df['SMILES_pair']:
            smiles1, smiles2 = pair.split(',')
            train_smiles.add(smiles1)
            train_smiles.add(smiles2)
        
        val_smiles = set()
        for pair in val_df['SMILES_pair']:
            smiles1, smiles2 = pair.split(',')
            val_smiles.add(smiles1)
            val_smiles.add(smiles2)
        
        if test_smile in train_smiles:
            print(f"WARNING: Test SMILES {test_smile} found in training set!")
        #else:
            #print(f"Verification: Test SMILES {test_smile} not found in training set.")
        
        if test_smile in val_smiles:
            print(f"WARNING: Test SMILES {test_smile} found in validation set!")
        #else:
            #print(f"Verification: Test SMILES {test_smile} not found in validation set.")
        
        best_val_loss = float('inf')
        epochs_no_improve = 0
        
        for epoch in range(num_epochs):
            # Training code (unchanged)
        

            model.train()
            total_loss = 0
            
            train_df = train_df.sample(frac=1).reset_index(drop=True)
            
            for i in range(0, len(train_df), batch_size):
                batch_df = train_df.iloc[i:i+batch_size]
                
                smile_pairs = batch_df.SMILES_pair.values
                y_val = batch_df[targets].values
                
                smiles_list1, smiles_list2 = zip(*[pair.split(',') for pair in smile_pairs])
                
                x_atom1, x_bonds1, x_atom_index1, x_bond_index1, x_mask1, _ = get_smiles_array(smiles_list1, feature_dicts)
                x_atom2, x_bonds2, x_atom_index2, x_bond_index2, x_mask2, _ = get_smiles_array(smiles_list2, feature_dicts)
                
                x_atom1, x_bonds1 = torch.Tensor(x_atom1).to(device), torch.Tensor(x_bonds1).to(device)
                x_atom_index1, x_bond_index1 = torch.LongTensor(x_atom_index1).to(device), torch.LongTensor(x_bond_index1).to(device)
                x_mask1 = torch.Tensor(x_mask1).to(device)
                
                x_atom2, x_bonds2 = torch.Tensor(x_atom2).to(device), torch.Tensor(x_bonds2).to(device)
                x_atom_index2, x_bond_index2 = torch.LongTensor(x_atom_index2).to(device), torch.LongTensor(x_bond_index2).to(device)
                x_mask2 = torch.Tensor(x_mask2).to(device)
                
                output_tuple = model(x_atom1, x_bonds1, x_atom_index1, x_bond_index1, x_mask1,
                                     x_atom2, x_bonds2, x_atom_index2, x_bond_index2, x_mask2)
                
                predictions = output_tuple[2]
                
                optimizer.zero_grad()
                loss = loss_function(predictions, torch.Tensor(y_val).to(device))
                loss.backward()
                optimizer.step()
                
                total_loss += loss.item()
            
            avg_loss = total_loss / (len(train_df) // batch_size + 1)
            #print(f'Epoch {epoch+1}/{num_epochs} done. Average Loss: {avg_loss}')
            
            # Validation step
            val_loss, val_r2 = validate(model, val_df, feature_dicts, loss_function, device, batch_size)
            #print(f'Validation Loss: {val_loss}, Validation R2: {val_r2}')
            

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                epochs_no_improve = 0

                # Delete the previous best model if it exists
                if best_model_path and os.path.exists(best_model_path):
                    os.remove(best_model_path)

                # Save the new best model
                unique_id = str(uuid.uuid4())[:8]
                best_model_path = f'/notebooks/GNN/saved_models/best_model_fold_{fold}_{unique_id}.pth'
                torch.save(model.state_dict(), best_model_path)
                #print(f'New best model found for fold {fold} with validation loss: {best_val_loss}')
            else:
                epochs_no_improve += 1

            if epochs_no_improve == patience:
                print(f'Early stopping triggered after {epoch+1} epochs')
                break
        

        # Evaluate on the test set for this fold
        model.load_state_dict(torch.load(f'/notebooks/GNN/saved_models/best_model_fold_{fold}_{unique_id}.pth'))
        #print(f'Created: best_model_fold_{fold}_{unique_id}.pth')
        test_loss, test_r2, true_values, predictions, result_dict = validate(model, test_df, feature_dicts, loss_function, device, batch_size, return_predictions=True, test_smile=test_smile)
        
        # Calculate MSE for this fold
        fold_mse = mean_squared_error(true_values, predictions)
        test_mses.append(fold_mse)

        final_dic.update(result_dict)
        
        all_true_values.extend(true_values)
        all_predictions.extend(predictions)
        
        fold_results.append((best_val_loss, test_loss, test_r2, fold_mse))  
        print(f'Fold {fold} completed. Best validation loss: {best_val_loss}, Test loss: {test_loss}, Test R2: {test_r2}')
        os.remove(f'/notebooks/GNN/saved_models/best_model_fold_{fold}_{unique_id}.pth') 
        #print(f' Removed: best_model_fold_{fold}_{unique_id}.pth')
    overall_r2 = r2_score(all_true_values, all_predictions)
    overall_mse = mean_squared_error(all_true_values, all_predictions)

    return fold_results, overall_r2, final_dic, test_r2, test_mses, overall_mse

# Usage
#fold_results, overall_r2, final_dic, test_r2s, test_mses, overall_mse = train_and_evaluate(model, df, feature_dicts, optimizer, loss_function, num_epochs=2, batch_size=32, patience=30)



In [10]:
import numpy as np
from sklearn.model_selection import ParameterSampler
from joblib import parallel_backend
import torch
from tqdm import tqdm

# Assuming you've already imported your functions
# from your_module import prepare_model_and_data_for_relative_learning, train_and_evaluate

# Define the parameter distributions for random search
param_distributions = {
    'radius': [2, 3, 4, 5, 6],
    'fingerprint_dim': [30, 70, 150, 210, 300],
    'p_dropout': [0.1, 0.2, 0.3, 0.4, 0.5],
    'weight_decay': [2, 3, 4, 5, 6],
    'learning_rate': [2, 3, 4, 5],
    'patience': [5, 10, 20, 30]
}

# Fixed parameters
fixed_params = {
    'T': 2,
    'batch_size': 32,
    'epochs': 1
}

def run_single_trial(params, raw_filename):
    model, optimizer, loss_function, df, feature_data, _, _ = prepare_model_and_data_for_relative_learning(
        raw_filename=raw_filename,
        radius=params['radius'],
        fingerprint_dim=params['fingerprint_dim'],
        p_dropout=params['p_dropout'],
        weight_decay=params['weight_decay'],
        learning_rate=params['learning_rate'],
        **fixed_params
    )
    
    _, overall_r2, _, _, _, _ = train_and_evaluate(
        model, df, feature_dicts, optimizer, loss_function,
        num_epochs=fixed_params['epochs'],
        batch_size=fixed_params['batch_size'],
        patience=params['patience']
    )
    
    return overall_r2, params

# Prepare your data
raw_filename = "/notebooks/data/cal_abs.csv"  # Replace with your actual filename

# Number of random trials
n_iter = 2

# Generate parameter combinations
param_list = list(ParameterSampler(param_distributions, n_iter=n_iter, random_state=42))

# Run the random search
results = []

with parallel_backend('loky'):
    if torch.cuda.is_available():
        device = torch.device("cuda:0")
        torch.cuda.set_device(device)
    
    for params in tqdm(param_list, desc="Running trials"):
        r2, params = run_single_trial(params, raw_filename)
        results.append((r2, params))

# Find the best parameters
best_r2, best_params = max(results, key=lambda x: x[0])

print("Best parameters found: ", best_params)
print("Best R2 score: ", best_r2)

# You can also store all results if needed
all_results = sorted(results, key=lambda x: x[0], reverse=True)


Running trials:   0%|          | 0/2 [00:00<?, ?it/s]

Fold 1 completed. Best validation loss: 5.635076665878296, Test loss: 4.1505599816640215, Test R2: 0.029641969671639026
Fold 2 completed. Best validation loss: 5.570839905738831, Test loss: 7.04884918530782, Test R2: 0.002023366468092591
Fold 3 completed. Best validation loss: 5.61629056930542, Test loss: 4.443106810251872, Test R2: 0.01299699667665477
Fold 4 completed. Best validation loss: 5.644105339050293, Test loss: 4.762562274932861, Test R2: -0.018409481408618894
Fold 5 completed. Best validation loss: 5.70118989944458, Test loss: 2.898355404535929, Test R2: 0.012180526248680101
Fold 6 completed. Best validation loss: 5.663296699523926, Test loss: 3.2562517325083413, Test R2: 0.013252926615628102
Fold 7 completed. Best validation loss: 5.721239972114563, Test loss: 3.045872608820597, Test R2: 0.0184058570787963
Fold 8 completed. Best validation loss: 5.525984835624695, Test loss: 5.460179567337036, Test R2: 0.004403649719329619
Fold 9 completed. Best validation loss: 5.706569814

Running trials:  50%|█████     | 1/2 [03:58<03:58, 238.58s/it]

Fold 40 completed. Best validation loss: 5.623189783096313, Test loss: 6.834034442901611, Test R2: 0.0010939257743358827
Fold 1 completed. Best validation loss: 5.678927206993103, Test loss: 4.268939971923828, Test R2: -0.0001918503185710796
Fold 2 completed. Best validation loss: 5.582794737815857, Test loss: 6.985739231109619, Test R2: -0.00022266787356173268
Fold 3 completed. Best validation loss: 5.630367422103882, Test loss: 4.558041334152222, Test R2: -0.00029625318978313
Fold 4 completed. Best validation loss: 5.720040225982666, Test loss: 4.6571173667907715, Test R2: -0.0002735001369049417
Fold 5 completed. Best validation loss: 5.751363921165466, Test loss: 2.9346593221028647, Test R2: -0.0004853171831239872
Fold 6 completed. Best validation loss: 5.7096027612686155, Test loss: 3.284690340360006, Test R2: 2.082668315352143e-05
Fold 7 completed. Best validation loss: 5.758512878417969, Test loss: 3.082818865776062, Test R2: -0.0003477072582845492
Fold 8 completed. Best validati

Running trials: 100%|██████████| 2/2 [06:09<00:00, 184.71s/it]

Fold 40 completed. Best validation loss: 5.624364733695984, Test loss: 6.756328264872233, Test R2: -0.0003208720677890242
Best parameters found:  {'weight_decay': 2, 'radius': 6, 'patience': 20, 'p_dropout': 0.3, 'learning_rate': 4, 'fingerprint_dim': 210}
Best R2 score:  0.0060701272767401115


In [ ]:
import torch
from sklearn.model_selection import ParameterSampler
from joblib import Parallel, delayed
import os

# Define the parameter distributions for random search
param_distributions = {
    'radius': [4, 5, 6, 7, 8],
    'fingerprint_dim': [30, 70, 150, 210, 300],
    'p_dropout': [0.1, 0.2, 0.3, 0.4, 0.5],
    'weight_decay': [2, 3, 4, 5, 6],
    'learning_rate': [2, 3, 4, 5],
    'patience': [5, 10, 20, 30]
}

# Fixed parameters
fixed_params = {
    'T': 2,
    'batch_size': 32,
    'epochs': 100
}

def run_single_trial(params):
    # Print the process ID to verify parallel execution
    print(f"Running trial with params: {params} on process ID: {os.getpid()}", flush=True)
    
    model, optimizer, loss_function, df, feature_data, _, _ = prepare_model_and_data_for_relative_learning(
        raw_filename=raw_filename,
        radius=params['radius'],
        fingerprint_dim=params['fingerprint_dim'],
        p_dropout=params['p_dropout'],
        weight_decay=params['weight_decay'],
        learning_rate=params['learning_rate'],
        **fixed_params
    )
    
    _, overall_r2, _, _, _, _ = train_and_evaluate(
        model, df, feature_dicts, optimizer, loss_function,
        num_epochs=fixed_params['epochs'],
        batch_size=fixed_params['batch_size'],
        patience=params['patience']
    )
    
    return overall_r2, params

# Prepare your data     
raw_filename = "/notebooks/data/cal_abs.csv" 

# Number of random trials
n_iter = 5

# Generate parameter combinations
param_list = list(ParameterSampler(param_distributions, n_iter=n_iter, random_state=42))

# Run the random search in parallel using joblib with verbose output
if torch.cuda.is_available():
    device = torch.device("cuda:0")
    torch.cuda.set_device(device)

results = Parallel(n_jobs=-1, verbose=10)(delayed(run_single_trial)(params) for params in param_list)

# Find the best parameters
best_r2, best_params = max(results, key=lambda x: x[0])

print("Best parameters found: ", best_params)
print("Best R2 score: ", best_r2)

# You can also store all results if needed
all_results = sorted(results, key=lambda x: x[0], reverse=True)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.


Running trial with params: {'weight_decay': 2, 'radius': 6, 'patience': 20, 'p_dropout': 0.4, 'learning_rate': 3, 'fingerprint_dim': 30} on process ID: 102
Running trial with params: {'weight_decay': 2, 'radius': 8, 'patience': 20, 'p_dropout': 0.3, 'learning_rate': 4, 'fingerprint_dim': 210} on process ID: 101
Running trial with params: {'weight_decay': 3, 'radius': 7, 'patience': 30, 'p_dropout': 0.2, 'learning_rate': 4, 'fingerprint_dim': 150} on process ID: 111
Running trial with params: {'weight_decay': 2, 'radius': 7, 'patience': 30, 'p_dropout': 0.4, 'learning_rate': 4, 'fingerprint_dim': 150} on process ID: 110
Running trial with params: {'weight_decay': 6, 'radius': 5, 'patience': 10, 'p_dropout': 0.3, 'learning_rate': 5, 'fingerprint_dim': 150} on process ID: 112
